In [1]:
# https://www.kaggle.com/c/house-prices-advanced-regression-techniques

In [1]:
! wget -O ~/datasets/train.csv \
https://raw.githubusercontent.com/erkansirin78/datasets/master/house-prices-dataset/train.csv

--2022-01-16 11:30:55--  https://raw.githubusercontent.com/erkansirin78/datasets/master/house-prices-dataset/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 460676 (450K) [text/plain]
Saving to: ‘/home/train/datasets/train.csv’

100%[======================================>] 460,676     2.38MB/s   in 0.2s   

2022-01-16 11:30:56 (2.38 MB/s) - ‘/home/train/datasets/train.csv’ saved [460676/460676]



In [3]:
! hdfs dfs -put -f ~/datasets/train.csv /user/train/datasets/house-prices

In [8]:
#! hdfs dfs -mkdir /user/train/datasets/house-prices

In [4]:
! hdfs dfs -ls /user/train/datasets/house-prices

-rw-r--r--   1 train supergroup     460676 2022-01-16 11:33 /user/train/datasets/house-prices


In [2]:
import findspark

findspark.init("/opt/manual/spark")

from pyspark.sql import SparkSession, functions as F
import pandas as pd

spark = SparkSession.builder \
.appName("House Price Regression") \
.master("yarn") \
.config("spark.sql.shuffle.partitions", "2") \
.getOrCreate()

## Read data

In [5]:
df = spark.read.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("/user/train/datasets/house-prices")

In [6]:
df.cache()

DataFrame[Id: int, MSSubClass: int, MSZoning: string, LotFrontage: string, LotArea: int, Street: string, Alley: string, LotShape: string, LandContour: string, Utilities: string, LotConfig: string, LandSlope: string, Neighborhood: string, Condition1: string, Condition2: string, BldgType: string, HouseStyle: string, OverallQual: int, OverallCond: int, YearBuilt: int, YearRemodAdd: int, RoofStyle: string, RoofMatl: string, Exterior1st: string, Exterior2nd: string, MasVnrType: string, MasVnrArea: string, ExterQual: string, ExterCond: string, Foundation: string, BsmtQual: string, BsmtCond: string, BsmtExposure: string, BsmtFinType1: string, BsmtFinSF1: int, BsmtFinType2: string, BsmtFinSF2: int, BsmtUnfSF: int, TotalBsmtSF: int, Heating: string, HeatingQC: string, CentralAir: string, Electrical: string, 1stFlrSF: int, 2ndFlrSF: int, LowQualFinSF: int, GrLivArea: int, BsmtFullBath: int, BsmtHalfBath: int, FullBath: int, HalfBath: int, BedroomAbvGr: int, KitchenAbvGr: int, KitchenQual: string

In [7]:
print(len(df.columns))

81


## Explore the data

In [8]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [9]:
df.limit(20).toPandas()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65,8450,Pave,NA,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NA,Attchd,2003,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NA,NA,NA,0,2,2008,WD,Normal,208500
1,2,20,RL,80,9600,Pave,NA,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,None,0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NA,NA,NA,0,5,2007,WD,Normal,181500
2,3,60,RL,68,11250,Pave,NA,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NA,NA,NA,0,9,2008,WD,Normal,223500
3,4,70,RL,60,9550,Pave,NA,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,None,0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NA,NA,NA,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84,14260,Pave,NA,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NA,NA,NA,0,12,2008,WD,Normal,250000
5,6,50,RL,85,14115,Pave,NA,IR1,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,1Fam,1.5Fin,5,5,1993,1995,Gable,CompShg,VinylSd,VinylSd,None,0,TA,TA,Wood,Gd,TA,No,GLQ,732,Unf,0,64,796,GasA,Ex,Y,SBrkr,796,566,0,1362,1,0,1,1,1,1,TA,5,Typ,0,NA,Attchd,1993,Unf,2,480,TA,TA,Y,40,30,0,320,0,0,NA,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75,10084,Pave,NA,Reg,Lvl,AllPub,Inside,Gtl,Somerst,Norm,Norm,1Fam,1Story,8,5,2004,2005,Gable,CompShg,VinylSd,VinylSd,Stone,186,Gd,TA,PConc,Ex,TA,Av,GLQ,1369,Unf,0,317,1686,GasA,Ex,Y,SBrkr,1694,0,0,1694,1,0,2,0,3,1,Gd,7,Typ,1,Gd,Attchd,2004,RFn,2,636,TA,TA,Y,255,57,0,0,0,0,NA,NA,NA,0,8,2007,WD,Normal,307000
7,8,60,RL,NA,10382,Pave,NA,IR1,Lvl,AllPub,Corner,Gtl,NWAmes,PosN,Norm,1Fam,2Story,7,6,1973,1973,Gable,CompShg,HdBoard,HdBoard,Stone,240,TA,TA,CBlock,Gd,TA,Mn,ALQ,859,BLQ,32,216,1107,GasA,Ex,Y,SBrkr,1107,983,0,2090,1,0,2,1,3,1,TA,7,Typ,2,TA,Attchd,1973,RFn,2,484,TA,TA,Y,235,204,228,0,0,0,NA,NA,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51,6120,Pave,NA,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Artery,Norm,1Fam,1.5Fin,7,5,1931,1950,Gable,CompShg,BrkFace,Wd Shng,None,0,TA,TA,BrkTil,TA,TA,No,Unf,0,Unf,0,952,952,GasA,Gd,Y,FuseF,1022,752,0,1774,0,0,2,0,2,2,TA,8,Min1,2,TA,Detchd,1931,Unf,2,468,Fa,TA,Y,90,0,205,0,0,0,NA,NA,NA,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50,7420,Pave,NA,Reg,Lvl,AllPub,Corner,Gtl,BrkSide,Artery,Artery,2fmCon,1.5Unf,5,6,1939,1950,Gable,CompShg,MetalSd,MetalSd,None,0,TA,TA,BrkTil,TA,TA,No,GLQ,851,Unf,0,140,991,GasA,Ex,Y,SBrkr,1077,0,0,1077,1,0,1,0,2,2,TA,5,Typ,2,TA,Attchd,1939,RFn,1,205,Gd,TA,Y,0,4,0,0,0,

In [10]:
df.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- MSSubClass: integer (nullable = true)
 |-- MSZoning: string (nullable = true)
 |-- LotFrontage: string (nullable = true)
 |-- LotArea: integer (nullable = true)
 |-- Street: string (nullable = true)
 |-- Alley: string (nullable = true)
 |-- LotShape: string (nullable = true)
 |-- LandContour: string (nullable = true)
 |-- Utilities: string (nullable = true)
 |-- LotConfig: string (nullable = true)
 |-- LandSlope: string (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Condition1: string (nullable = true)
 |-- Condition2: string (nullable = true)
 |-- BldgType: string (nullable = true)
 |-- HouseStyle: string (nullable = true)
 |-- OverallQual: integer (nullable = true)
 |-- OverallCond: integer (nullable = true)
 |-- YearBuilt: integer (nullable = true)
 |-- YearRemodAdd: integer (nullable = true)
 |-- RoofStyle: string (nullable = true)
 |-- RoofMatl: string (nullable = true)
 |-- Exterior1st: string (nullable = true)
 |--

## Null check

In [11]:
avg_dict = {}

In [12]:
df_count = df.count()

In [13]:
df_count

1460

In [14]:
# Seperate categorical and numeric null including columns
cat_nulls = []

num_nulls = []

In [15]:
def null_count(df, col_name):
    nc = df.filter( 
        (F.col(col_name).isNull()) | 
        (F.col(col_name) == "") | 
        (F.col(col_name) == "NA")
    ).count()
    
    return nc

In [18]:
for col_name in df.dtypes:
    
    nc = null_count(df, col_name[0])
    
    
    if(  nc > 0 ):
        print("{} {} type has {} null values, % {}".format(col_name[0], col_name[1], nc, (nc/df_count)))
        if col_name[1] == 'string':
            cat_nulls.append(col_name[0])
        else:
            num_nulls.append(col_name[0])

LotFrontage string type has 259 null values, % 0.1773972602739726
Alley string type has 1369 null values, % 0.9376712328767123
MasVnrType string type has 8 null values, % 0.005479452054794521
MasVnrArea string type has 8 null values, % 0.005479452054794521
BsmtQual string type has 37 null values, % 0.025342465753424658
BsmtCond string type has 37 null values, % 0.025342465753424658
BsmtExposure string type has 38 null values, % 0.026027397260273973
BsmtFinType1 string type has 37 null values, % 0.025342465753424658
BsmtFinType2 string type has 38 null values, % 0.026027397260273973
Electrical string type has 1 null values, % 0.0006849315068493151
FireplaceQu string type has 690 null values, % 0.4726027397260274
GarageType string type has 81 null values, % 0.05547945205479452
GarageYrBlt string type has 81 null values, % 0.05547945205479452
GarageFinish string type has 81 null values, % 0.05547945205479452
GarageQual string type has 81 null values, % 0.05547945205479452
GarageCond strin

In [19]:
print(cat_nulls)

['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']


In [20]:
print(num_nulls)

[]


In [30]:
# Alley, PoolQC, Fence, MiscFeature have too much nulls. I decide to discard these cols.

## Group columns

In [24]:
categoric_cols = []

numeric_cols = []

discarted_cols = ['Id','GarageYrBlt','Neighborhood','Alley', 'PoolQC', 'Fence', 'MiscFeature']
# GarageYrBlt(98), Neighborhood (25) too many categories 
# Alley, PoolQC, Fence, MiscFeature have too much nulls. I decide to discard these cols

label_col = ['SalePrice']

In [25]:
# column seperation
for col_name in df.dtypes:
    if (col_name[0] not in label_col+discarted_cols):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [26]:
print(categoric_cols)
print(len(categoric_cols))

['MSZoning', 'LotFrontage', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'SaleType', 'SaleCondition']
40


In [27]:
print(numeric_cols)
print(len(numeric_cols))

['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']
33


In [28]:
print(discarted_cols)
print(len(discarted_cols))

['Id', 'GarageYrBlt', 'Neighborhood', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']
7


In [29]:
print(label_col)
print(len(label_col))

['SalePrice']
1


### column check

In [30]:
len(df.columns) == (len(categoric_cols)  + len(numeric_cols) + len(discarted_cols) + len(label_col))

True

## Dealing mis-inferred columns

In [10]:
# Examine data and check if inferred type is ok. WE can use an excel sheet to take notes.

# LotFrontage	string	0 inferred as str but doubt it could be 
# MasVnrArea	string	0


### Examine and Impute LotFrontage

In [31]:
df.groupBy("LotFrontage").count().toPandas()

,LotFrontage,count
0,65,44
1,80,69
2,68,19
3,60,143
4,85,40
5,70,70
6,72,17
7,66,15
8,57,12
9,44,9


In [32]:
df.select("LotFrontage").filter("LotFrontage == 'NA'").limit(5).toPandas()

,LotFrontage
0,NA
1,NA
2,NA
3,NA
4,NA


In [13]:
# LotFrontage cant be a string must be int. Spark must be infer it as string because there is NA values.

# I decided to fill with avg

In [33]:
df.selectExpr("AVG(LotFrontage)").show()

+--------------------------------+
|avg(CAST(LotFrontage AS DOUBLE))|
+--------------------------------+
|               70.04995836802665|
+--------------------------------+



### Examine and Impute MasVnrArea

In [34]:
df.groupBy("MasVnrArea").count().toPandas()

,MasVnrArea,count
0,186,3
1,212,2
2,380,1
3,640,1
4,200,6
5,246,4
6,650,2
7,412,1
8,272,3
9,178,4


In [35]:
df.select("MasVnrArea").filter("MasVnrArea == 'NA'").limit(5).toPandas()

,MasVnrArea
0,NA
1,NA
2,NA
3,NA
4,NA


In [17]:
# MasVnrArea cant be a string must be int. Spark must be infer it as string because there is NA values.

# I decided to fill with avg

In [36]:
df.selectExpr("AVG(MasVnrArea)").show()

+-------------------------------+
|avg(CAST(MasVnrArea AS DOUBLE))|
+-------------------------------+
|             103.68526170798899|
+-------------------------------+



#### Impute and cast dtypes

In [37]:
df1 = df.withColumn("LotFrontage", F.when(F.col("LotFrontage") == 'NA', 70) \
                                .otherwise(F.col("LotFrontage"))) \
                    .withColumn("LotFrontage", F.col("LotFrontage").cast("int")) \
                    .withColumn("MasVnrArea", F.when(F.col("MasVnrArea") == 'NA', 103) \
                                                    .otherwise(F.col("MasVnrArea"))) \
                    .withColumn("MasVnrArea", F.col("MasVnrArea").cast("int"))

In [38]:
df1.select("LotFrontage").filter("LotFrontage == 'NA'").limit(5).toPandas()

,LotFrontage


In [39]:
df1.select("MasVnrArea").filter("MasVnrArea == 'NA'").limit(5).toPandas()

,MasVnrArea


In [40]:
df.select("LotFrontage","MasVnrArea").printSchema()

root
 |-- LotFrontage: string (nullable = true)
 |-- MasVnrArea: string (nullable = true)



In [41]:
df1.select("LotFrontage","MasVnrArea").printSchema()

root
 |-- LotFrontage: integer (nullable = true)
 |-- MasVnrArea: integer (nullable = true)



## Examine categoricals

In [42]:
for col_name in categoric_cols:
    print(col_name)
    df1.groupBy(col_name).count().orderBy(F.desc("count")).show()

MSZoning
+--------+-----+
|MSZoning|count|
+--------+-----+
|      RL| 1151|
|      RM|  218|
|      FV|   65|
|      RH|   16|
| C (all)|   10|
+--------+-----+

LotFrontage
+-----------+-----+
|LotFrontage|count|
+-----------+-----+
|         70|  329|
|         60|  143|
|         80|   69|
|         50|   57|
|         75|   53|
|         65|   44|
|         85|   40|
|         78|   25|
|         90|   23|
|         21|   23|
|         64|   19|
|         24|   19|
|         68|   19|
|         73|   18|
|         79|   17|
|         55|   17|
|         72|   17|
|         63|   17|
|        100|   16|
|         74|   15|
+-----------+-----+
only showing top 20 rows

Street
+------+-----+
|Street|count|
+------+-----+
|  Pave| 1454|
|  Grvl|    6|
+------+-----+

LotShape
+--------+-----+
|LotShape|count|
+--------+-----+
|     Reg|  925|
|     IR1|  484|
|     IR2|   41|
|     IR3|   10|
+--------+-----+

LandContour
+-----------+-----+
|LandContour|count|
+-----------+-----+
|  

+-------------+-----+
|SaleCondition|count|
+-------------+-----+
|       Normal| 1198|
|      Partial|  125|
|      Abnorml|  101|
|       Family|   20|
|       Alloca|   12|
|      AdjLand|    4|
+-------------+-----+



## Which columns will be hot coded?

In [43]:
one_hot_cols = []

In [44]:
for col_name in categoric_cols:
    dist_count = df1.select(col_name).distinct().count()
    print(col_name, dist_count)
    if  dist_count > 2:
        one_hot_cols.append(col_name)

MSZoning 5
LotFrontage 110
Street 2
LotShape 4
LandContour 4
Utilities 2
LotConfig 5
LandSlope 3
Condition1 9
Condition2 8
BldgType 5
HouseStyle 8
RoofStyle 6
RoofMatl 8
Exterior1st 15
Exterior2nd 16
MasVnrType 5
MasVnrArea 328
ExterQual 4
ExterCond 5
Foundation 6
BsmtQual 5
BsmtCond 5
BsmtExposure 5
BsmtFinType1 7
BsmtFinType2 7
Heating 6
HeatingQC 5
CentralAir 2
Electrical 6
KitchenQual 4
Functional 7
FireplaceQu 6
GarageType 7
GarageFinish 4
GarageQual 6
GarageCond 6
PavedDrive 3
SaleType 9
SaleCondition 6


In [45]:
print(one_hot_cols)

['MSZoning', 'LotFrontage', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'SaleType', 'SaleCondition']


# StringIndexer

In [46]:
from pyspark.ml.feature import StringIndexer

In [47]:
my_dict = {}

string_indexer_objs = []

string_indexer_output_names = []

ohe_input_names = []

ohe_output_names = []

for col_name in categoric_cols:
    my_dict[col_name+"_index_obj"] = StringIndexer() \
                                    .setHandleInvalid("skip") \
                                    .setInputCol(col_name) \
                                    .setOutputCol(col_name+"_indexed")
    
    string_indexer_objs.append(my_dict.get(col_name+"_index_obj"))
    string_indexer_output_names.append(col_name+"_indexed")
    if col_name in one_hot_cols:
        ohe_input_names.append(col_name+"_indexed")
        ohe_output_names.append(col_name+"_ohe")

In [48]:
print(string_indexer_objs)
print(len(string_indexer_objs))

[StringIndexer_108024981562, StringIndexer_12f7f8e99729, StringIndexer_d571df2e5e97, StringIndexer_0076a3ad3a44, StringIndexer_16e9b47f8f84, StringIndexer_cc921588d807, StringIndexer_ad4184a5a952, StringIndexer_590b0d48e946, StringIndexer_e57c7c2e8b30, StringIndexer_e3982c9c9504, StringIndexer_fc140019f552, StringIndexer_2d78750af93a, StringIndexer_cfd62ad2f72d, StringIndexer_d62c68472dfb, StringIndexer_f74645a402b6, StringIndexer_36525a446e0e, StringIndexer_9b4b4612f340, StringIndexer_fc44694fd260, StringIndexer_65ab684d47c3, StringIndexer_0efc7bd0ab11, StringIndexer_13c677cd5e51, StringIndexer_7b4c6874422b, StringIndexer_3d9d5d462094, StringIndexer_d51118094fde, StringIndexer_af27c7d5c2b1, StringIndexer_dbfa6977a49b, StringIndexer_a7f51a672761, StringIndexer_d23668ce7ba1, StringIndexer_1c2860110f61, StringIndexer_4adf87609dc2, StringIndexer_640ffb800be6, StringIndexer_226832d89878, StringIndexer_420e194f5eb6, StringIndexer_a7de0cb55c8b, StringIndexer_265392b15ba4, StringIndexer_b4707

In [49]:
print(string_indexer_output_names)
print(len(string_indexer_output_names))

['MSZoning_indexed', 'LotFrontage_indexed', 'Street_indexed', 'LotShape_indexed', 'LandContour_indexed', 'Utilities_indexed', 'LotConfig_indexed', 'LandSlope_indexed', 'Condition1_indexed', 'Condition2_indexed', 'BldgType_indexed', 'HouseStyle_indexed', 'RoofStyle_indexed', 'RoofMatl_indexed', 'Exterior1st_indexed', 'Exterior2nd_indexed', 'MasVnrType_indexed', 'MasVnrArea_indexed', 'ExterQual_indexed', 'ExterCond_indexed', 'Foundation_indexed', 'BsmtQual_indexed', 'BsmtCond_indexed', 'BsmtExposure_indexed', 'BsmtFinType1_indexed', 'BsmtFinType2_indexed', 'Heating_indexed', 'HeatingQC_indexed', 'CentralAir_indexed', 'Electrical_indexed', 'KitchenQual_indexed', 'Functional_indexed', 'FireplaceQu_indexed', 'GarageType_indexed', 'GarageFinish_indexed', 'GarageQual_indexed', 'GarageCond_indexed', 'PavedDrive_indexed', 'SaleType_indexed', 'SaleCondition_indexed']
40


In [50]:
print(ohe_input_names)
print(len(ohe_input_names))

['MSZoning_indexed', 'LotFrontage_indexed', 'LotShape_indexed', 'LandContour_indexed', 'LotConfig_indexed', 'LandSlope_indexed', 'Condition1_indexed', 'Condition2_indexed', 'BldgType_indexed', 'HouseStyle_indexed', 'RoofStyle_indexed', 'RoofMatl_indexed', 'Exterior1st_indexed', 'Exterior2nd_indexed', 'MasVnrType_indexed', 'MasVnrArea_indexed', 'ExterQual_indexed', 'ExterCond_indexed', 'Foundation_indexed', 'BsmtQual_indexed', 'BsmtCond_indexed', 'BsmtExposure_indexed', 'BsmtFinType1_indexed', 'BsmtFinType2_indexed', 'Heating_indexed', 'HeatingQC_indexed', 'Electrical_indexed', 'KitchenQual_indexed', 'Functional_indexed', 'FireplaceQu_indexed', 'GarageType_indexed', 'GarageFinish_indexed', 'GarageQual_indexed', 'GarageCond_indexed', 'PavedDrive_indexed', 'SaleType_indexed', 'SaleCondition_indexed']
37


In [51]:
print(ohe_output_names)
print(len(ohe_output_names))

['MSZoning_ohe', 'LotFrontage_ohe', 'LotShape_ohe', 'LandContour_ohe', 'LotConfig_ohe', 'LandSlope_ohe', 'Condition1_ohe', 'Condition2_ohe', 'BldgType_ohe', 'HouseStyle_ohe', 'RoofStyle_ohe', 'RoofMatl_ohe', 'Exterior1st_ohe', 'Exterior2nd_ohe', 'MasVnrType_ohe', 'MasVnrArea_ohe', 'ExterQual_ohe', 'ExterCond_ohe', 'Foundation_ohe', 'BsmtQual_ohe', 'BsmtCond_ohe', 'BsmtExposure_ohe', 'BsmtFinType1_ohe', 'BsmtFinType2_ohe', 'Heating_ohe', 'HeatingQC_ohe', 'Electrical_ohe', 'KitchenQual_ohe', 'Functional_ohe', 'FireplaceQu_ohe', 'GarageType_ohe', 'GarageFinish_ohe', 'GarageQual_ohe', 'GarageCond_ohe', 'PavedDrive_ohe', 'SaleType_ohe', 'SaleCondition_ohe']
37


# OnehotEncoder

In [52]:
from pyspark.ml.feature import OneHotEncoder

In [53]:
encoder = OneHotEncoder() \
.setInputCols(ohe_input_names) \
.setOutputCols(ohe_output_names)

# VectorAssembler

In [54]:
from pyspark.ml.feature import VectorAssembler

In [55]:
not_to_hot_coded = list(set(string_indexer_output_names).difference(set(ohe_input_names)))

print(not_to_hot_coded)

['Street_indexed', 'CentralAir_indexed', 'Utilities_indexed']


In [56]:
if len(categoric_cols) == (len(not_to_hot_coded)+len(ohe_output_names)):
    print("Columns verified")
else: print("Columns are not verified")

Columns verified


In [57]:
assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(numeric_cols+not_to_hot_coded+ohe_output_names) \
.setOutputCol('unscaled_features')

# Feature Scaling

In [58]:
from pyspark.ml.feature import StandardScaler

In [59]:
scaler = StandardScaler().setInputCol("unscaled_features").setOutputCol("features")

# Estimator: GBTRegressor

In [60]:
from pyspark.ml.regression import GBTRegressor

In [61]:
estimator = GBTRegressor() \
.setFeaturesCol("features") \
.setLabelCol(label_col[0])

# Pipeline

In [62]:
from pyspark.ml import Pipeline

In [63]:
pipeline_obj = Pipeline().setStages(string_indexer_objs + [encoder, assembler, scaler, estimator])

# Split test train

In [64]:
train_df, test_df = df1.randomSplit([.8, .2], seed=142)

In [65]:
print(train_df.count())
print(test_df.count())

1163
297


# Train the model

In [66]:
pipeline_model = pipeline_obj.fit(train_df)

# Prediction

In [67]:
transformed_df = pipeline_model.transform(test_df)

In [68]:
transformed_df.select(label_col[0],'prediction').show(truncate=False)

+---------+------------------+
|SalePrice|prediction        |
+---------+------------------+
|140000   |142819.45505559075|
|129900   |181260.73913747413|
|129500   |143667.43203812029|
|157000   |147628.38666081207|
|154000   |154034.8447437488 |
|207500   |198858.34941529512|
|165500   |156469.6529620314 |
|239686   |332147.6342314651 |
|127000   |122233.02852182915|
|110000   |97096.71038983157 |
|385000   |800864.6274948628 |
|101000   |80217.64564052563 |
|204750   |196922.67063002128|
|214000   |230162.27051307308|
|198900   |198244.16840665747|
|180000   |154959.12495811147|
|320000   |334702.7149972438 |
|87000    |99962.2344761866  |
|244000   |204465.314537233  |
|180000   |163197.7409823176 |
+---------+------------------+
only showing top 20 rows



# Evaluate the model

In [69]:
from pyspark.ml.evaluation import RegressionEvaluator

In [73]:
evaluator = RegressionEvaluator(labelCol=label_col[0], metricName='rmse')

In [74]:
evaluator.evaluate(transformed_df)

36965.09068704467

In [75]:
evaluator.getMetricName()

'rmse'

In [77]:
spark.stop()